# Lab 1 — GR00T Fine-Tuning: Training & Inference

**Prerequisite:** Run `Lab0_Build_Container.ipynb` first to build the ECR image. 
**Prerequisite:** Run `Lab0_Build_Container.ipynb` first to build the ECR image.

---

## What this notebook does

| Section | What happens |
|---------|-------------|
| 0. Setup | Resolve AWS resources, set HuggingFace token |
| 1. Explore Dataset | Visualise the 27 teleop episodes (parquet, video, tasks) |
| 2. Upload to S3 | Sync dataset to S3 for the training job |
| 3. Launch Training | Submit a SageMaker Training Job on `ml.g5.2xlarge` |
| 4. Monitor & Download | Poll job status, download the trained checkpoint |
| 5. Inspect Results | Plot training loss curve |
| 6. Run Inference | Open-loop evaluation: predicted vs ground-truth actions |
| 7. Understand Output | Interpret the inference plots and MSE numbers |

## What the training does
Fine-tunes GR00T N1.6-3B on 27 UR3 pick-and-place demonstrations.
Only the **projector** layer is trained (200-step smoke test). Only ~6% of
parameters are trainable, which fits in a single A10G.

## What inference produces
Comparison plots (PNG) of predicted vs ground-truth joint velocity curves.
**Not a video of the robot moving** — that requires Isaac Sim (Lab 2) or real hardware.
The plots validate that the model is making sensible predictions.

## Estimated cost
- 200 steps on `ml.g5.2xlarge`: **~$1-2, ~30 min**
- 2000 steps (properly trained): **~$10-15, ~3 hrs**


## 0 — Setup

In [ ]:
import boto3, json, os, subprocess, sys, time
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# Resolve repo root regardless of where notebook is opened
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "aws-physical-ai-toolchain" and REPO_ROOT != REPO_ROOT.parent:
 REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

LAB1_DIR = REPO_ROOT / "notebooks" / "lab1"
DATASET_LOCAL = REPO_ROOT / "training" / "data" / "ur3_lerobot_dataset"
SDK_DIR = LAB1_DIR / "isaac-groot" # cloned Isaac-GR00T SDK

REGION = boto3.session.Session().region_name or "us-west-2"
ACCOUNT = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/service-role/AmazonSageMaker-ExecutionRole-20260607T114524"
ECR_URI = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/physical-ai/groot-training:latest"
DATASET_PREFIX = "groot-data/ur3"

print(f"Repo: {REPO_ROOT}")
print(f"Region: {REGION} | Account: {ACCOUNT}")
print(f"Bucket: {BUCKET}")
print(f"ECR: {ECR_URI}")
print(f"Dataset: {DATASET_LOCAL} (exists: {DATASET_LOCAL.exists()})")
print(f"SDK: {SDK_DIR} (exists: {SDK_DIR.exists()})")


In [ ]:
# ── HuggingFace Token ───────────────────────────────────────────────────
# Token is read from notebooks/lab1/.env — set it once, never paste into notebooks.
#
# Setup (one time):
# Create notebooks/lab1/.env with: HF_TOKEN=hf_your_token
#
# Get token: https://huggingface.co/settings/tokens
# Accept license: https://huggingface.co/nvidia/GR00T-N1.6-3B

from dotenv import load_dotenv
load_dotenv(LAB1_DIR / '.env')

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
 print(f' HF_TOKEN loaded from .env ({HF_TOKEN[:8]}...)')
else:
 print(' HF_TOKEN not set.')
 print(' Create notebooks/lab1/.env with: HF_TOKEN=hf_your_token')


## 1 — Explore the Dataset

In [ ]:
# ── Dataset overview ──────────────────────────────────────────────────────
info = json.loads((DATASET_LOCAL / "meta" / "info.json").read_text())
print("Dataset info:")
print(f" Robot: {info['robot_type']}")
print(f" Episodes: {info['total_episodes']} (human teleop demonstrations)")
print(f" Frames: {info['total_frames']} (timesteps across all episodes)")
print(f" FPS: {info['fps']} (camera + telemetry rate)")
print()

# Episodes
print("First 5 episodes:")
with open(DATASET_LOCAL / "meta" / "episodes.jsonl") as f:
 episodes = [json.loads(l) for l in f]
for ep in episodes[:5]:
 print(f" ep{ep['episode_index']:02d}: {ep['length']:3d} frames | {ep['tasks'][0][:70]}")
print(f" ... ({len(episodes)} total)")
print()

# Sample parquet row
import pandas as pd
pf = sorted((DATASET_LOCAL / "data" / "chunk-000").glob("*.parquet"))[0]
df = pd.read_parquet(pf)
print(f"Parquet episode_000000: {len(df)} frames × {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
row0 = df.iloc[0]
print(f"\nFrame 0:")
print(f" observation.state = {np.round(row0['observation.state'], 3)}")
print(f" joints[0:6] = UR3 joint angles in radians")
print(f" [6] = gripper (0=open, 1=closed)")
print(f" action = {np.round(row0['action'], 4)}")
print(f" [0:3] = Cartesian velocity (vx, vy, vz) × dt")
print(f" [3:6] = Rotation velocity (rx, ry, rz) × dt")
print(f" [6] = Gripper target position")


In [ ]:
# ── Plot joint trajectories for episode 0 ────────────────────────────────
actions = np.stack(df["action"].values)
states = np.stack(df["observation.state"].values)
labels = ["vx","vy","vz","rx","ry","rz","gripper"]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for j in range(6):
 axes[0].plot(actions[:, j], label=labels[j])
axes[0].plot(actions[:, 6], "k--", lw=1.5, label="gripper")
axes[0].set_ylabel("Command value"); axes[0].legend(ncol=4, fontsize=8)
axes[0].set_title("Episode 0 — Commanded Actions (what the human operator sent)", fontsize=11)

for j in range(6):
 axes[1].plot(states[:, j], label=f"j{j+1}")
axes[1].set_ylabel("Joint angle (rad)"); axes[1].set_xlabel("Frame (5 fps)")
axes[1].legend(ncol=6, fontsize=8)
axes[1].set_title("Episode 0 — Joint State (where the robot arm actually was)", fontsize=11)

plt.tight_layout(); plt.show()
print(f"Episode length: {len(df)} frames = {len(df)/5:.1f} seconds of motion")


In [ ]:
# ── Show wrist camera frames from 4 different episodes ───────────────────
import cv2

video_dir = DATASET_LOCAL / "videos" / "chunk-000" / "observation.images.wrist"
vids = sorted(video_dir.glob("*.mp4"))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for row_idx, frame_num in enumerate([5, 30]):
 for col_idx, vid in enumerate(vids[:4]):
 cap = cv2.VideoCapture(str(vid))
 cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
 ret, frame = cap.read(); cap.release()
 if ret:
 axes[row_idx, col_idx].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
 ep_idx = int(vid.stem.split("_")[-1])
 task = episodes[ep_idx]["tasks"][0][:50] if ep_idx < len(episodes) else ""
 axes[row_idx, col_idx].set_title(f"Ep {ep_idx} | frame {frame_num}\n{task}", fontsize=7)
 axes[row_idx, col_idx].axis("off")

plt.suptitle("Wrist Camera Views — What the Robot Sees (480×640 RGB at 5fps)", fontsize=12)
plt.tight_layout(); plt.show()
print("These images are the model's primary visual input during training and inference.")


## 1b — Data Preparation (Zarr → LeRobot v2)

---

### What GR00T requires — LeRobot v2 format

The Isaac-GR00T SDK (`launch_finetune.py`) only accepts data in **LeRobot v2 format**.
It cannot read raw Zarr files or any other format directly.

LeRobot v2 consists of:
```
dataset/
├── data/chunk-000/episode_*.parquet ← numerical data (state, action, task_index per frame)
├── videos/chunk-000/observation.images.wrist/*.mp4 ← wrist camera video
└── meta/
 ├── info.json ← dataset shape and feature definitions
 ├── modality.json ← maps column names to GR00T's state/action/video slots
 ├── tasks.jsonl ← task vocabulary: task_index → text instruction
 ├── episodes.jsonl ← per-episode description and length
 ├── stats.json ← mean/std per feature (generated by SDK before training)
 └── relative_stats.json ← relative action stats (generated by SDK before training)
```

---

### Input data required — Raw Zarr teleop recordings

`convert_zarr_to_lerobot.py` reads raw UR3 teleop recordings in **Zarr format**.
Each episode is a folder containing:
```
episode_NNN_.../
├── observations/joints ← 6 joint angles (radians, ~10Hz)
├── observations/gripper_position ← gripper state (0–255)
├── observations/timestamps ← telemetry timestamps
├── images/wrist ← wrist camera RGB frames (5Hz)
├── images/wrist_timestamps ← camera timestamps
├── commands.json ← URScript speedl() commands from Xbox controller
└── zarr.json ← metadata: task_name, task_description, camera_hz
```

**Expected location:** `training/data/ur3_episodes/episodes/` 
(change `ZARR_EPISODES_DIR` in the cell below if your data is elsewhere)

**How to get the data:**
- If provided as a zip (`ur3_episodes_001_027.zip`): extract it to `training/data/ur3_episodes/`
- If you have your own UR3 teleop recordings: place episode folders in the same structure
- This lab ships with 27 pre-recorded pick-and-place episodes

---

### The conversion script

**Script:** `lab1/convert_zarr_to_lerobot.py` 
**Source:** Part of `aws-physical-ai-toolchain` — not from the NVIDIA GR00T repo 
**What it does:** Aligns camera frames (5Hz) with telemetry (10Hz) by timestamp,
converts URScript `speedl()` velocity commands into 7D action vectors,
and writes LeRobot v2 parquet + MP4 + metadata files.

Set `REGENERATE_DATASET = True` in the next cell to run the conversion. 
Leave `False` (default) if the dataset already exists at `DATASET_LOCAL`.


In [ ]:
# ── Data preparation settings ────────────────────────────────────────────
# Set REGENERATE_DATASET = True to re-run the Zarr→LeRobot conversion.
# Set to False (default) to use the existing dataset at DATASET_LOCAL.

REGENERATE_DATASET = False

# Path to raw Zarr teleop episodes.
# Change this if your data is in a different location.
# Expected structure: <ZARR_EPISODES_DIR>/episode_NNN_<task_name>/
ZARR_EPISODES_DIR = REPO_ROOT / "training" / "data" / "ur3_episodes" / "episodes"

# Path where the converted LeRobot v2 dataset will be written (and read from).
# DATASET_LOCAL is defined in cell [2] above as training/data/ur3_lerobot_dataset/
CONVERT_SCRIPT = LAB1_DIR / "convert_zarr_to_lerobot.py"

# ── Run conversion if requested ──────────────────────────────────────────
if REGENERATE_DATASET:
 if not ZARR_EPISODES_DIR.exists():
 print(f" Zarr episodes not found at: {ZARR_EPISODES_DIR}")
 print()
 print("To get the raw data:")
 print(" Option 1 — Extract from zip:")
 print(f" unzip ur3_episodes_001_027.zip -d {ZARR_EPISODES_DIR.parent}")
 print()
 print(" Option 2 — Use your own UR3 teleop recordings:")
 print(f" Copy episode folders to: {ZARR_EPISODES_DIR}")
 print(" Each folder must contain: observations/, images/, commands.json, zarr.json")
 else:
 n_eps = len([p for p in ZARR_EPISODES_DIR.iterdir() if p.is_dir()])
 print(f"Found {n_eps} Zarr episodes at: {ZARR_EPISODES_DIR}")
 print(f"Output: {DATASET_LOCAL}")
 print()
 print("Running conversion (~2-3 min for 27 episodes)...")

 result = subprocess.run([
 sys.executable, str(CONVERT_SCRIPT),
 "--episodes-dir", str(ZARR_EPISODES_DIR),
 "--output-dir", str(DATASET_LOCAL),
 ], cwd=str(REPO_ROOT))

 if result.returncode == 0:
 print()
 print(" Conversion complete.")
 # Delete stale stats so SDK regenerates with correct task_index
 for sf in ["stats.json", "relative_stats.json"]:
 p = DATASET_LOCAL / "meta" / sf
 if p.exists():
 p.unlink()
 print(f" Deleted stale {sf} (will regenerate during training job)")
 else:
 print(f" Conversion failed (exit code {result.returncode})")
else:
 if DATASET_LOCAL.exists() and (DATASET_LOCAL / "meta" / "info.json").exists():
 info = json.loads((DATASET_LOCAL / "meta" / "info.json").read_text())
 print(f" Using existing dataset:")
 print(f" Location: {DATASET_LOCAL}")
 print(f" Episodes: {info['total_episodes']} | Frames: {info['total_frames']}")
 print()
 print(" Set REGENERATE_DATASET = True to reconvert from Zarr source data.")
 else:
 print(f" No dataset found at: {DATASET_LOCAL}")
 print()
 print("Options:")
 print(" 1. Set REGENERATE_DATASET = True and ensure Zarr episodes are at ZARR_EPISODES_DIR")
 print(f" 2. Copy a LeRobot v2 dataset directly to: {DATASET_LOCAL}")


## 2 — Upload Dataset to S3

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

# Check if already uploaded (stats.json is generated last, good sentinel)
resp = s3.list_objects_v2(
 Bucket=BUCKET, Prefix=f"{DATASET_PREFIX}/dataset/meta/stats.json"
)
if resp.get("KeyCount", 0) > 0:
 print(f" Dataset already at s3://{BUCKET}/{DATASET_PREFIX}/dataset/")
 print(" Skipping upload (delete from S3 to force re-upload).")
else:
 print("Uploading dataset to S3 (parquet + videos + meta)...")
 result = subprocess.run(
 ["aws", "s3", "sync", str(DATASET_LOCAL),
 f"s3://{BUCKET}/{DATASET_PREFIX}/dataset/", "--region", REGION],
 capture_output=True, text=True
 )
 if result.returncode == 0:
 print(f" Uploaded to s3://{BUCKET}/{DATASET_PREFIX}/dataset/")
 else:
 print("ERROR:", result.stderr[:300])

# Confirm key files are present
for key in ["meta/info.json", "meta/stats.json", "meta/modality.json"]:
 try:
 s3.head_object(Bucket=BUCKET, Key=f"{DATASET_PREFIX}/dataset/{key}")
 print(f" {key}")
 except:
 print(f" MISSING: {key}")


## 3 — Launch SageMaker Training Job

The training job runs `train_entrypoint_v2.py` inside the ECR container.
That script calls the Isaac-GR00T SDK's `launch_finetune.py` with these flags:

```
--no-tune-visual # freeze visual encoder (SigLIP2) — default is already False
--no-tune-diffusion-model # freeze diffusion action head — saves memory on single GPU
--tune-projector # train the projector bridge layer (default True, ~201M params)
```

**Source:** `notebooks/lab1/isaac-groot/gr00t/experiment/launch_finetune.py` 
**Entrypoint:** `notebooks/lab1/container/train_entrypoint_v2.py`

Change `MAX_STEPS` below:
- `200` = smoke test, proves the pipeline works, ~30 min, ~$1
- `2000` = proper training, model starts to generalise, ~3 hrs, ~$10


In [ ]:
MAX_STEPS = 200 # change to 2000 for a properly trained model

sm = boto3.client("sagemaker", region_name=REGION)
job_name = f"groot-ur3-{int(time.time())}"

sm.create_training_job(
 TrainingJobName=job_name,
 RoleArn=ROLE_ARN,
 AlgorithmSpecification={
 "TrainingImage": ECR_URI,
 "TrainingInputMode": "File",
 },
 HyperParameters={
 "base_model": "nvidia/GR00T-N1.6-3B",
 "max_steps": str(MAX_STEPS),
 "batch_size": "8", # 2 per GPU × 4 GPUs on g5.12xlarge
 "grad_accum": "4", # effective batch = 32
 "learning_rate": "1e-4",
 "hf_token": os.environ.get("HF_TOKEN", ""),
 },
 InputDataConfig=[{
 "ChannelName": "training",
 "DataSource": {"S3DataSource": {
 "S3DataType": "S3Prefix",
 "S3Uri": f"s3://{BUCKET}/{DATASET_PREFIX}/dataset/",
 "S3DataDistributionType": "FullyReplicated",
 }},
 }],
 OutputDataConfig={"S3OutputPath": f"s3://{BUCKET}/{DATASET_PREFIX}/output/"},
 ResourceConfig={
 "InstanceType": "ml.g5.12xlarge", # 4× A10G = 96 GB VRAM, ~$7.09/hr
 "InstanceCount": 1,
 "VolumeSizeInGB": 100,
 },
 StoppingCondition={"MaxRuntimeInSeconds": 7200},
 Environment={"HF_TOKEN": os.environ.get("HF_TOKEN", "")},
)

print(f" Job submitted: {job_name}")
print(f" Instance: ml.g5.12xlarge (A10G 24 GB, ~$1.62/hr)")
print(f" Steps: {MAX_STEPS} | Est. duration: ~{MAX_STEPS // 6} min")
print(f" Output: s3://{BUCKET}/{DATASET_PREFIX}/output/{job_name}/output/model.tar.gz")
print()
print("Run the next cell to monitor progress.")


## 4 — Monitor Job & Download Model

Re-run the status cell until `Status: Completed`.

In [ ]:
# ── Poll status — re-run to refresh ──────────────────────────────────────
desc = sm.describe_training_job(TrainingJobName=job_name)
status = desc["TrainingJobStatus"]
secondary = desc.get("SecondaryStatus", "")

print(f"Job: {job_name}")
print(f"Status: {status} / {secondary}")
print()
if status not in ("Completed", "Failed"):
 print("Typical progression:")
 print(" Pending → Downloading (pulls ECR image + dataset) → Training → Uploading → Completed")
 print("\nRe-run this cell in ~2 min to check again.")
elif status == "Failed":
 print(" Failed:", desc.get("FailureReason", "unknown")[:200])
 print()
 print(f"CloudWatch logs:")
 print(f" https://console.aws.amazon.com/cloudwatch/home?region={REGION}"
 f"#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs"
 f"/log-events/{job_name}")


In [ ]:
# ── Download model artifact (run after status == Completed) ──────────────
import shutil

desc = sm.describe_training_job(TrainingJobName=job_name)
assert desc["TrainingJobStatus"] == "Completed", f"Job not done yet: {desc['TrainingJobStatus']} / {desc.get('SecondaryStatus')}"

model_s3 = desc["ModelArtifacts"]["S3ModelArtifacts"]
duration = (desc["TrainingEndTime"] - desc["TrainingStartTime"]).seconds // 60
MODEL_DIR = Path.home() / "groot-trained-model"

print(f"Job completed in {duration} min")
print(f"Downloading model artifact from S3 (~12-25 GB)...")

if MODEL_DIR.exists():
 shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir()
tar_path = MODEL_DIR / "model.tar.gz"

bucket_name = model_s3.split("/")[2]
key = "/".join(model_s3.split("/")[3:])
s3.download_file(bucket_name, key, str(tar_path))
print(f"Downloaded: {tar_path.stat().st_size / 1e9:.1f} GB")

subprocess.run(["tar", "-xzf", str(tar_path), "-C", str(MODEL_DIR)], check=True)

checkpoints = sorted(MODEL_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
print(f"\n Extracted to: {MODEL_DIR}")
print(f"Checkpoints: {[c.name for c in checkpoints]}")


## 5 — Inspect Training Loss

In [ ]:
latest_ckpt = sorted(MODEL_DIR.glob("checkpoint-*"),
 key=lambda p: int(p.name.split("-")[1]))[-1]
state = json.loads((latest_ckpt / "trainer_state.json").read_text())

steps = [e["step"] for e in state["log_history"] if "loss" in e]
losses = [e["loss"] for e in state["log_history"] if "loss" in e]

print(f"Training loss: {losses[0]:.4f} (start) → {losses[-1]:.4f} (end)")
print(f"Reduction: {(losses[0]-losses[-1])/losses[0]*100:.1f}%")
print()
print("What these numbers mean:")
print(" ~1.3 = model barely better than random noise (expected at start)")
print(" ~1.1 = very early learning signal (expected after 200 steps)")
print(" ~0.5 = model tracks general motion direction (needs ~1000 steps)")
print(" ~0.1 = model closely replicates human demos (needs ~2000+ steps)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, losses, "b-o", markersize=4)
ax.set_xlabel("Training Step"); ax.set_ylabel("Loss (diffusion denoising error)")
ax.set_title(f"GR00T Fine-Tuning Loss — projector only, {steps[-1]} steps")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 6 — Open-Loop Inference

We feed pre-recorded dataset frames through the fine-tuned model and compare its
predicted actions to what the human actually did.

**What this is:**
- The model sees a wrist camera frame + current joint state + task description
- It predicts a 16-step action chunk (what joint velocities to apply next)
- We compare that to the ground-truth actions from the human demo

**What this is NOT:**
- Not a video of the robot moving
- Not the robot actually running autonomously
- Not a real-time control loop

For an actual robot video you need Lab 2 (Isaac Sim) or a physical UR3.

**The SDK script used:** `notebooks/lab1/isaac-groot/gr00t/eval/open_loop_eval.py` 
**Source:** https://github.com/NVIDIA/Isaac-GR00T/blob/main/gr00t/eval/open_loop_eval.py


In [ ]:
PLOTS_DIR = Path.home() / "groot-inference-plots"
PLOTS_DIR.mkdir(exist_ok=True)

env = dict(os.environ)
env["PYTHONPATH"] = str(SDK_DIR)
env["USE_TF"] = "0"
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Running open-loop evaluation on 3 trajectories...")
print(f"Using checkpoint: {latest_ckpt.name}")
print()

result = subprocess.run([
 sys.executable,
 str(SDK_DIR / "gr00t" / "eval" / "open_loop_eval.py"),
 "--model-path", str(latest_ckpt),
 "--dataset-path", str(DATASET_LOCAL),
 "--embodiment-tag","new_embodiment",
 "--traj-ids", "0", "3", "6", # 3 episodes spread across the dataset
 "--action-horizon","16",
 "--steps", "200",
 "--denoising-steps","4",
 "--modality-keys", "arm", "gripper",
 "--save-plot-path", str(PLOTS_DIR),
], env=env, capture_output=True, text=True, cwd=str(REPO_ROOT))

# Print MSE/MAE results
for line in result.stdout.splitlines():
 if any(k in line for k in ["MSE", "MAE", "Average", "trajectory", "Using", "Running"]):
 print(line.replace("INFO:root:", "").strip())

if result.returncode != 0:
 print("\nERROR:", result.stderr[-400:])


## 7 — View Inference Plots

In [ ]:
import IPython.display as ipd

# The SDK saves one PNG per trajectory
plot_files = sorted(PLOTS_DIR.glob("*.png"))

if plot_files:
 print(f"Generated {len(plot_files)} comparison plot(s).")
 print()
 print("How to read each plot:")
 print(" - X axis: frame number (timestep)")
 print(" - Y axis: action value (joint velocity or gripper position)") 
 print(" - BLUE line: ground truth (what the human actually did)")
 print(" - RED/ORANGE line: model prediction (what GR00T would do)")
 print(" - Closer curves = better model")
 print()
 for pf in plot_files:
 print(f" {pf.name}")
 display(ipd.Image(str(pf), width=950))
else:
 # SDK may save to a single combined file
 single = Path.home() / "groot-inference-plots.png"
 if single.exists():
 display(ipd.Image(str(single), width=950))
 else:
 print("No plot files found. Check the output above for errors.")


## 8 — Summary & Next Steps

### What you've done
- Fine-tuned GR00T N1.6-3B on 27 UR3 pick-and-place demonstrations
- Trained the projector layer (~201M params) on your robot's action space
- Validated the model predicts sensible actions on held-out trajectories

### Interpreting the inference plots
| What you see | What it means |
|-------------|--------------|
| Predicted curve roughly follows GT shape | Model learned the general task motion |
| Predicted curve is flat / near-zero | Model needs more training steps |
| Predicted curve is noisy but directionally correct | Expected at 200 steps |
| Predicted curve closely matches GT | Well-trained model (needs 2000+ steps) |

### To improve the model
Run with `MAX_STEPS = 2000` and add `--tune-diffusion-model` flag in `train_entrypoint_v2.py`:
```python
# In container/train_entrypoint_v2.py, remove this line:
"--no-tune-diffusion-model",
```
Then rebuild the container (Lab0) and rerun training.

### To see the robot actually moving
- **Lab 2:** Isaac Sim workstation — visualise and debug policies in simulation
- **Lab 4:** Isaac Lab RL — refine the policy with reinforcement learning
- **Lab 5:** Edge deployment — run on a real UR3 arm

### Model artifact location
```
S3: s3://{BUCKET}/groot-data/ur3/output/{job_name}/output/model.tar.gz
Local: ~/groot-trained-model/checkpoint-{N}/
```
